In [1]:
import pandas as pd

df = pd.read_csv("Chennai.csv")
df_hyd = pd.read_csv("Hyderabad.csv")
df_Kol = pd.read_csv("Kolkata.csv")
df_Mum = pd.read_csv("Mumbai.csv")
df_newd = pd.read_csv("New_Delhi.csv")

datasets = [df, df_hyd, df_Kol, df_Mum, df_newd]

for data in datasets:
    print(data.shape)
    print(data.columns.tolist())
    print()

(5597, 5)
['Location', 'Time', 'Temperature_C', 'Humidity_Percent', 'Pressure_hPa']

(3879, 5)
['Location', 'Time', 'Temperature_C', 'Humidity_Percent', 'Pressure_hPa']

(5590, 5)
['Location', 'Time', 'Temperature_C', 'Humidity_Percent', 'Pressure_hPa']

(5365, 5)
['Location', 'Time', 'Temperature_C', 'Humidity_Percent', 'Pressure_hPa']

(1862, 5)
['Location', 'Time', 'Temperature_C', 'Humidity_Percent', 'Pressure_hPa']



In [2]:
for data in datasets:
    data["DateTime"] = pd.to_datetime(data["Time"])

for data in datasets:
    print(data["Location"].iloc[0])
    print(data[["Time", "DateTime"]].head(5))
    print()

Chennai
                  Time            DateTime
0  2025-01-01 00:00:00 2025-01-01 00:00:00
1  2025-01-01 01:00:00 2025-01-01 01:00:00
2  2025-01-01 02:00:00 2025-01-01 02:00:00
3  2025-01-01 03:00:00 2025-01-01 03:00:00
4  2025-01-01 04:00:00 2025-01-01 04:00:00

Hyderabad
                  Time            DateTime
0  2025-01-01 00:00:00 2025-01-01 00:00:00
1  2025-01-01 02:00:00 2025-01-01 02:00:00
2  2025-01-01 03:00:00 2025-01-01 03:00:00
3  2025-01-01 04:00:00 2025-01-01 04:00:00
4  2025-01-01 06:00:00 2025-01-01 06:00:00

Kolkata
                  Time            DateTime
0  2025-01-01 00:00:00 2025-01-01 00:00:00
1  2025-01-01 01:00:00 2025-01-01 01:00:00
2  2025-01-01 02:00:00 2025-01-01 02:00:00
3  2025-01-01 03:00:00 2025-01-01 03:00:00
4  2025-01-01 04:00:00 2025-01-01 04:00:00

Mumbai
                  Time            DateTime
0  2025-01-01 00:00:00 2025-01-01 00:00:00
1  2025-01-01 01:00:00 2025-01-01 01:00:00
2  2025-01-01 02:00:00 2025-01-01 02:00:00
3  2025-01-01 03:0

In [3]:
def resample_station(data):
    data = data.copy()
    data = data.set_index("DateTime")

    result = data.resample("3h").agg({
        "Location": "first",
        "Temperature_C": "first",
        "Humidity_Percent": "first",
        "Pressure_hPa": "first"
    })

    result = result.reset_index()

    return result


df_3h = resample_station(df)
df_hyd_3h = resample_station(df_hyd)
df_Kol_3h = resample_station(df_Kol)
df_Mum_3h = resample_station(df_Mum)
df_newd_3h = resample_station(df_newd)

In [4]:
for data in [df_3h, df_hyd_3h, df_Kol_3h, df_Mum_3h, df_newd_3h]:
    print(data["Location"].iloc[0], data.shape)
    print(data.head())
    print()

Chennai (1888, 5)
             DateTime Location  Temperature_C  Humidity_Percent  Pressure_hPa
0 2025-01-01 00:00:00  Chennai           24.0         94.719963        1012.9
1 2025-01-01 03:00:00  Chennai           24.8         84.970838        1015.4
2 2025-01-01 06:00:00  Chennai           29.0         69.685553        1015.1
3 2025-01-01 09:00:00  Chennai           30.0         58.581385        1012.5
4 2025-01-01 12:00:00  Chennai           30.0         58.581385        1012.5

Hyderabad (1888, 5)
             DateTime   Location  Temperature_C  Humidity_Percent  \
0 2025-01-01 00:00:00  Hyderabad           19.0         88.745728   
1 2025-01-01 03:00:00  Hyderabad           20.6         85.594668   
2 2025-01-01 06:00:00  Hyderabad           27.6         47.981939   
3 2025-01-01 09:00:00  Hyderabad           28.6         49.175064   
4 2025-01-01 12:00:00  Hyderabad           27.0         50.986772   

   Pressure_hPa  
0        1015.0  
1        1016.7  
2        1015.9  
3     

In [5]:
for data in [df_3h, df_hyd_3h, df_Kol_3h, df_Mum_3h, df_newd_3h]:
    print(
        data["Location"].iloc[0],
        "\nMissing values:\n",
        data.isna().sum(),
        "\n"
    )

Chennai 
Missing values:
 DateTime             0
Location             5
Temperature_C        5
Humidity_Percent     5
Pressure_hPa        39
dtype: int64 

Hyderabad 
Missing values:
 DateTime             0
Location            34
Temperature_C       34
Humidity_Percent    34
Pressure_hPa        47
dtype: int64 

Kolkata 
Missing values:
 DateTime             0
Location             5
Temperature_C        5
Humidity_Percent     5
Pressure_hPa        28
dtype: int64 

Mumbai 
Missing values:
 DateTime             0
Location             5
Temperature_C        5
Humidity_Percent     5
Pressure_hPa        21
dtype: int64 

New_Delhi 
Missing values:
 DateTime             0
Location            31
Temperature_C       31
Humidity_Percent    35
Pressure_hPa        32
dtype: int64 



In [6]:
def remove_empty_rows(data):
    return data.dropna(
        subset=["Temperature_C", "Humidity_Percent", "Pressure_hPa"],
        how="all"
    ).reset_index(drop=True)


df_3h = remove_empty_rows(df_3h)
df_hyd_3h = remove_empty_rows(df_hyd_3h)
df_Kol_3h = remove_empty_rows(df_Kol_3h)
df_Mum_3h = remove_empty_rows(df_Mum_3h)
df_newd_3h = remove_empty_rows(df_newd_3h)

In [7]:
for data in [df_3h, df_hyd_3h, df_Kol_3h, df_Mum_3h, df_newd_3h]:
    print(data["Location"].iloc[0], data.shape)
    print(data.isna().sum())
    print()

Chennai (1883, 5)
DateTime             0
Location             0
Temperature_C        0
Humidity_Percent     0
Pressure_hPa        34
dtype: int64

Hyderabad (1854, 5)
DateTime             0
Location             0
Temperature_C        0
Humidity_Percent     0
Pressure_hPa        13
dtype: int64

Kolkata (1883, 5)
DateTime             0
Location             0
Temperature_C        0
Humidity_Percent     0
Pressure_hPa        23
dtype: int64

Mumbai (1883, 5)
DateTime             0
Location             0
Temperature_C        0
Humidity_Percent     0
Pressure_hPa        16
dtype: int64

New_Delhi (1857, 5)
DateTime            0
Location            0
Temperature_C       0
Humidity_Percent    4
Pressure_hPa        1
dtype: int64



In [8]:
datasets_3h = [
    df_3h,
    df_hyd_3h,
    df_Kol_3h,
    df_Mum_3h,
    df_newd_3h
]

for data in datasets_3h:
    data["Pressure_Missing"] = data["Pressure_hPa"].isna().astype(int)

In [9]:
for data in datasets_3h:
    data.set_index("DateTime", inplace=True)

    data["Pressure_hPa"] = data["Pressure_hPa"].interpolate(
        method="time",
        limit_direction="both"
    )

    data.reset_index(inplace=True)

In [10]:
for data in datasets_3h:
    print(data["Location"].iloc[0])
    print(data.isna().sum())
    print()

Chennai
DateTime            0
Location            0
Temperature_C       0
Humidity_Percent    0
Pressure_hPa        0
Pressure_Missing    0
dtype: int64

Hyderabad
DateTime            0
Location            0
Temperature_C       0
Humidity_Percent    0
Pressure_hPa        0
Pressure_Missing    0
dtype: int64

Kolkata
DateTime            0
Location            0
Temperature_C       0
Humidity_Percent    0
Pressure_hPa        0
Pressure_Missing    0
dtype: int64

Mumbai
DateTime            0
Location            0
Temperature_C       0
Humidity_Percent    0
Pressure_hPa        0
Pressure_Missing    0
dtype: int64

New_Delhi
DateTime            0
Location            0
Temperature_C       0
Humidity_Percent    4
Pressure_hPa        0
Pressure_Missing    0
dtype: int64



In [11]:
df_newd_3h["Humidity_Percent"] = (
    df_newd_3h["Humidity_Percent"].interpolate(
        method="linear",
        limit_direction="both"
    )
)

In [12]:
for data in datasets_3h:
    print(data["Location"].iloc[0])
    print(data.isna().sum())
    print()

Chennai
DateTime            0
Location            0
Temperature_C       0
Humidity_Percent    0
Pressure_hPa        0
Pressure_Missing    0
dtype: int64

Hyderabad
DateTime            0
Location            0
Temperature_C       0
Humidity_Percent    0
Pressure_hPa        0
Pressure_Missing    0
dtype: int64

Kolkata
DateTime            0
Location            0
Temperature_C       0
Humidity_Percent    0
Pressure_hPa        0
Pressure_Missing    0
dtype: int64

Mumbai
DateTime            0
Location            0
Temperature_C       0
Humidity_Percent    0
Pressure_hPa        0
Pressure_Missing    0
dtype: int64

New_Delhi
DateTime            0
Location            0
Temperature_C       0
Humidity_Percent    0
Pressure_hPa        0
Pressure_Missing    0
dtype: int64



In [13]:
final_df = pd.concat(
    datasets_3h,
    ignore_index=True
)

# Sort chronologically, with locations grouped naturally by timestamp
final_df = final_df.sort_values(
    ["DateTime", "Location"]
).reset_index(drop=True)

print(final_df.shape)
print(final_df.head(10))
print(final_df.isna().sum())

(9360, 6)
             DateTime   Location  Temperature_C  Humidity_Percent  \
0 2025-01-01 00:00:00    Chennai           24.0         94.719963   
1 2025-01-01 00:00:00  Hyderabad           19.0         88.745728   
2 2025-01-01 00:00:00    Kolkata           15.0         86.697136   
3 2025-01-01 00:00:00     Mumbai           22.0         85.735734   
4 2025-01-01 00:00:00  New_Delhi           10.4         92.273983   
5 2025-01-01 03:00:00    Chennai           24.8         84.970838   
6 2025-01-01 03:00:00  Hyderabad           20.6         85.594668   
7 2025-01-01 03:00:00    Kolkata           16.1         79.225550   
8 2025-01-01 03:00:00     Mumbai           22.6         85.795576   
9 2025-01-01 03:00:00  New_Delhi           10.0         86.791067   

   Pressure_hPa  Pressure_Missing  
0        1012.9                 0  
1        1015.0                 0  
2        1015.8                 0  
3        1013.0                 0  
4        1019.2                 0  
5        1015.

In [14]:
final_df.to_csv("SkyGuard_clean_3hourly.csv", index=False)

print("Saved successfully!")
print(final_df.shape)

Saved successfully!
(9360, 6)


In [17]:
import pandas as pd
import numpy as np

# Load the cleaned dataset
final_df = pd.read_csv("SkyGuard_clean_3hourly.csv")

# Convert DateTime back to datetime
final_df["DateTime"] = pd.to_datetime(final_df["DateTime"])

# Sort correctly within each station
final_df = final_df.sort_values(
    ["Location", "DateTime"]
).reset_index(drop=True)


# ============================================================
# 1. TIME FEATURES
# ============================================================

# Extract time components
final_df["Hour"] = final_df["DateTime"].dt.hour
final_df["DayOfYear"] = final_df["DateTime"].dt.dayofyear
final_df["Month"] = final_df["DateTime"].dt.month

# Cyclic encoding
final_df["Hour_Sin"] = np.sin(2 * np.pi * final_df["Hour"] / 24)
final_df["Hour_Cos"] = np.cos(2 * np.pi * final_df["Hour"] / 24)

final_df["DayOfYear_Sin"] = np.sin(
    2 * np.pi * final_df["DayOfYear"] / 365
)

final_df["DayOfYear_Cos"] = np.cos(
    2 * np.pi * final_df["DayOfYear"] / 365
)


# ============================================================
# 2. DIFFERENCE / RATE-OF-CHANGE FEATURES
# ============================================================

# IMPORTANT:
# These are calculated separately for each location.

grouped = final_df.groupby("Location")

final_df["Temperature_Diff"] = grouped["Temperature_C"].diff()
final_df["Humidity_Diff"] = grouped["Humidity_Percent"].diff()
final_df["Pressure_Diff"] = grouped["Pressure_hPa"].diff()

# Absolute magnitude of change
final_df["Temperature_AbsDiff"] = final_df["Temperature_Diff"].abs()
final_df["Humidity_AbsDiff"] = final_df["Humidity_Diff"].abs()
final_df["Pressure_AbsDiff"] = final_df["Pressure_Diff"].abs()


# ============================================================
#3. CORRECTED ROLLING FEATURES — PREVIOUS OBSERVATIONS ONLY
# ============================================================

window = 4

final_df["Temperature_RollingMean"] = grouped[
    "Temperature_C"
].transform(
    lambda x: x.shift(1).rolling(window).mean()
)

final_df["Humidity_RollingMean"] = grouped[
    "Humidity_Percent"
].transform(
    lambda x: x.shift(1).rolling(window).mean()
)

final_df["Pressure_RollingMean"] = grouped[
    "Pressure_hPa"
].transform(
    lambda x: x.shift(1).rolling(window).mean()
)

final_df["Temperature_RollingStd"] = grouped[
    "Temperature_C"
].transform(
    lambda x: x.shift(1).rolling(window).std()
)

final_df["Humidity_RollingStd"] = grouped[
    "Humidity_Percent"
].transform(
    lambda x: x.shift(1).rolling(window).std()
)

final_df["Pressure_RollingStd"] = grouped[
    "Pressure_hPa"
].transform(
    lambda x: x.shift(1).rolling(window).std()
)

# ============================================================
# 4. DEVIATION FROM ROLLING NORMAL
# ============================================================

final_df["Temperature_Deviation"] = (
    final_df["Temperature_C"]
    - final_df["Temperature_RollingMean"]
)

final_df["Humidity_Deviation"] = (
    final_df["Humidity_Percent"]
    - final_df["Humidity_RollingMean"]
)

final_df["Pressure_Deviation"] = (
    final_df["Pressure_hPa"]
    - final_df["Pressure_RollingMean"]
)


# ============================================================
# 5. Z-SCORE-LIKE LOCAL ANOMALY FEATURES
# ============================================================

final_df["Temperature_LocalZ"] = (
    final_df["Temperature_Deviation"]
    / (final_df["Temperature_RollingStd"] + 1e-6)
)

final_df["Humidity_LocalZ"] = (
    final_df["Humidity_Deviation"]
    / (final_df["Humidity_RollingStd"] + 1e-6)
)

final_df["Pressure_LocalZ"] = (
    final_df["Pressure_Deviation"]
    / (final_df["Pressure_RollingStd"] + 1e-6)
)


# ============================================================
# 6. REMOVE TEMPORARY COLUMNS
# ============================================================

final_df = final_df.drop(
    columns=[
        "Hour",
        "DayOfYear",
        "Month"
    ]
)


# ============================================================
# 7. HANDLE INITIAL ROLLING NaNs
# ============================================================

# First few observations of each station don't have
# enough previous observations for rolling statistics.

feature_columns = [
    "Temperature_Diff",
    "Humidity_Diff",
    "Pressure_Diff",
    "Temperature_AbsDiff",
    "Humidity_AbsDiff",
    "Pressure_AbsDiff",
    "Temperature_RollingMean",
    "Humidity_RollingMean",
    "Pressure_RollingMean",
    "Temperature_RollingStd",
    "Humidity_RollingStd",
    "Pressure_RollingStd",
    "Temperature_Deviation",
    "Humidity_Deviation",
    "Pressure_Deviation",
    "Temperature_LocalZ",
    "Humidity_LocalZ",
    "Pressure_LocalZ"
]

# Fill only feature-engineering NaNs
final_df[feature_columns] = (
    final_df.groupby("Location")[feature_columns]
    .transform(lambda x: x.bfill().ffill())
)


# ============================================================
# 8. FINAL SORT
# ============================================================

final_df = final_df.sort_values(
    ["DateTime", "Location"]
).reset_index(drop=True)


# ============================================================
# 9. CHECK RESULT
# ============================================================

print("Shape:", final_df.shape)

print("\nColumns:")
print(final_df.columns.tolist())

print("\nMissing values:")
print(final_df.isna().sum())

print("\nSample:")
print(final_df.head())

Shape: (9360, 28)

Columns:
['DateTime', 'Location', 'Temperature_C', 'Humidity_Percent', 'Pressure_hPa', 'Pressure_Missing', 'Hour_Sin', 'Hour_Cos', 'DayOfYear_Sin', 'DayOfYear_Cos', 'Temperature_Diff', 'Humidity_Diff', 'Pressure_Diff', 'Temperature_AbsDiff', 'Humidity_AbsDiff', 'Pressure_AbsDiff', 'Temperature_RollingMean', 'Humidity_RollingMean', 'Pressure_RollingMean', 'Temperature_RollingStd', 'Humidity_RollingStd', 'Pressure_RollingStd', 'Temperature_Deviation', 'Humidity_Deviation', 'Pressure_Deviation', 'Temperature_LocalZ', 'Humidity_LocalZ', 'Pressure_LocalZ']

Missing values:
DateTime                   0
Location                   0
Temperature_C              0
Humidity_Percent           0
Pressure_hPa               0
Pressure_Missing           0
Hour_Sin                   0
Hour_Cos                   0
DayOfYear_Sin              0
DayOfYear_Cos              0
Temperature_Diff           0
Humidity_Diff              0
Pressure_Diff              0
Temperature_AbsDiff        0


In [18]:
final_df.to_csv("SkyGuard_features.csv", index=False)

print("Saved!")
print(final_df.shape)

Saved!
(9360, 28)


In [19]:
model_features = [
    "Temperature_C",
    "Humidity_Percent",
    "Pressure_hPa",
    "Temperature_Diff",
    "Humidity_Diff",
    "Pressure_Diff",
    "Temperature_AbsDiff",
    "Humidity_AbsDiff",
    "Pressure_AbsDiff",
    "Temperature_Deviation",
    "Humidity_Deviation",
    "Pressure_Deviation",
    "Temperature_LocalZ",
    "Humidity_LocalZ",
    "Pressure_LocalZ",
    "Hour_Sin",
    "Hour_Cos",
    "DayOfYear_Sin",
    "DayOfYear_Cos",
    "Pressure_Missing"
]

X = final_df[model_features].copy()

print("Feature matrix shape:", X.shape)
print("Missing values:", X.isna().sum().sum())

Feature matrix shape: (9360, 20)
Missing values: 0


In [20]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# Store results
isolation_results = []

# Anomaly proportion we initially expect
# 2% of observations will be flagged
CONTAMINATION = 0.02

for location in final_df["Location"].unique():

    print(f"\nTraining Isolation Forest for {location}...")

    # Select one station
    station_df = final_df[
        final_df["Location"] == location
    ].copy()

    # Get features
    X_station = station_df[model_features].copy()

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_station)

    # Create Isolation Forest
    model = IsolationForest(
        n_estimators=300,
        contamination=CONTAMINATION,
        random_state=42,
        n_jobs=-1
    )

    # Train
    model.fit(X_scaled)

    # Predictions
    predictions = model.predict(X_scaled)

    # Convert:
    # Isolation Forest: 1 = normal, -1 = anomaly
    station_df["IF_Anomaly"] = (
        predictions == -1
    ).astype(int)

    # Anomaly score
    station_df["IF_Score"] = -model.decision_function(
        X_scaled
    )

    isolation_results.append(station_df)

# Combine all stations
if_results = pd.concat(
    isolation_results,
    ignore_index=True
)

# Sort chronologically
if_results = if_results.sort_values(
    ["DateTime", "Location"]
).reset_index(drop=True)

# Results
print("\n" + "=" * 50)
print("ISOLATION FOREST RESULTS")
print("=" * 50)

print("\nTotal observations:", len(if_results))

print(
    "Total anomalies:",
    if_results["IF_Anomaly"].sum()
)

print(
    "Anomaly percentage:",
    round(
        if_results["IF_Anomaly"].mean() * 100,
        2
    ),
    "%"
)

print("\nAnomalies by location:")
print(
    if_results.groupby("Location")[
        "IF_Anomaly"
    ].agg(["sum", "count", "mean"])
)


Training Isolation Forest for Chennai...

Training Isolation Forest for Hyderabad...

Training Isolation Forest for Kolkata...

Training Isolation Forest for Mumbai...

Training Isolation Forest for New_Delhi...

ISOLATION FOREST RESULTS

Total observations: 9360
Total anomalies: 190
Anomaly percentage: 2.03 %

Anomalies by location:
           sum  count      mean
Location                       
Chennai     38   1883  0.020181
Hyderabad   38   1854  0.020496
Kolkata     38   1883  0.020181
Mumbai      38   1883  0.020181
New_Delhi   38   1857  0.020463


In [21]:
print(
    if_results[
        if_results["IF_Anomaly"] == 1
    ][[
        "DateTime",
        "Location",
        "Temperature_C",
        "Humidity_Percent",
        "Pressure_hPa",
        "IF_Score"
    ]].head(20)
)

               DateTime   Location  Temperature_C  Humidity_Percent  \
11  2025-01-01 06:00:00  Hyderabad           27.6         47.981939   
50  2025-01-02 06:00:00    Chennai           29.4         53.245045   
51  2025-01-02 06:00:00  Hyderabad           26.2         42.895494   
91  2025-01-03 06:00:00  Hyderabad           27.4         26.904966   
93  2025-01-03 06:00:00     Mumbai           31.6         33.927743   
94  2025-01-03 06:00:00  New_Delhi           12.6         93.014695   
98  2025-01-03 09:00:00     Mumbai           29.8         23.892436   
99  2025-01-03 09:00:00  New_Delhi           20.0         60.795284   
133 2025-01-04 06:00:00     Mumbai           31.4         25.832231   
171 2025-01-05 06:00:00  Hyderabad           27.2         32.924112   
173 2025-01-05 06:00:00     Mumbai           29.4         37.493494   
212 2025-01-06 06:00:00    Kolkata           23.8         59.728866   
251 2025-01-07 06:00:00  Hyderabad           26.6         45.898404   
286 20

In [22]:
from pyod.models.ecod import ECOD
from pyod.models.copod import COPOD
from pyod.models.hbos import HBOS
from sklearn.preprocessing import StandardScaler

pyod_results = []

CONTAMINATION = 0.02

for location in final_df["Location"].unique():

    print(f"\nTraining PyOD models for {location}...")

    station_df = final_df[
        final_df["Location"] == location
    ].copy()

    X_station = station_df[model_features].copy()

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_station)

    # --------------------------------------------------------
    # ECOD
    # --------------------------------------------------------

    ecod = ECOD(
        contamination=CONTAMINATION
    )

    ecod.fit(X_scaled)

    station_df["ECOD_Anomaly"] = ecod.labels_
    station_df["ECOD_Score"] = ecod.decision_scores_

    # --------------------------------------------------------
    # COPOD
    # --------------------------------------------------------

    copod = COPOD(
        contamination=CONTAMINATION
    )

    copod.fit(X_scaled)

    station_df["COPOD_Anomaly"] = copod.labels_
    station_df["COPOD_Score"] = copod.decision_scores_

    # --------------------------------------------------------
    # HBOS
    # --------------------------------------------------------

    hbos = HBOS(
        contamination=CONTAMINATION
    )

    hbos.fit(X_scaled)

    station_df["HBOS_Anomaly"] = hbos.labels_
    station_df["HBOS_Score"] = hbos.decision_scores_

    pyod_results.append(station_df)


# Combine all stations
pyod_results = pd.concat(
    pyod_results,
    ignore_index=True
)

# Sort
pyod_results = pyod_results.sort_values(
    ["DateTime", "Location"]
).reset_index(drop=True)


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 60)
print("PYOD RESULTS")
print("=" * 60)

print("\nECOD anomalies:")
print(pyod_results["ECOD_Anomaly"].sum())

print("\nCOPOD anomalies:")
print(pyod_results["COPOD_Anomaly"].sum())

print("\nHBOS anomalies:")
print(pyod_results["HBOS_Anomaly"].sum())

print("\nAnomaly percentages:")

print(
    "ECOD:",
    round(pyod_results["ECOD_Anomaly"].mean() * 100, 2),
    "%"
)

print(
    "COPOD:",
    round(pyod_results["COPOD_Anomaly"].mean() * 100, 2),
    "%"
)

print(
    "HBOS:",
    round(pyod_results["HBOS_Anomaly"].mean() * 100, 2),
    "%"
)


Training PyOD models for Chennai...

Training PyOD models for Hyderabad...

Training PyOD models for Kolkata...

Training PyOD models for Mumbai...

Training PyOD models for New_Delhi...

PYOD RESULTS

ECOD anomalies:
190

COPOD anomalies:
190

HBOS anomalies:
190

Anomaly percentages:
ECOD: 2.03 %
COPOD: 2.03 %
HBOS: 2.03 %


In [23]:
# ============================================================
# MODEL AGREEMENT ANALYSIS
# ============================================================

# Merge Isolation Forest results with PyOD results
results = if_results.merge(
    pyod_results[
        [
            "DateTime",
            "Location",
            "ECOD_Anomaly",
            "ECOD_Score",
            "COPOD_Anomaly",
            "COPOD_Score",
            "HBOS_Anomaly",
            "HBOS_Score"
        ]
    ],
    on=["DateTime", "Location"],
    how="inner"
)

# ------------------------------------------------------------
# Count how many models flagged each observation
# ------------------------------------------------------------

results["Model_Agreement"] = (
    results["IF_Anomaly"]
    + results["ECOD_Anomaly"]
    + results["COPOD_Anomaly"]
    + results["HBOS_Anomaly"]
)

# ------------------------------------------------------------
# Distribution of agreement
# ------------------------------------------------------------

print("=" * 60)
print("MODEL AGREEMENT")
print("=" * 60)

print("\nNumber of models agreeing on anomaly:")

print(
    results["Model_Agreement"]
    .value_counts()
    .sort_index()
)

print("\nPercentage of dataset:")

print(
    (results["Model_Agreement"]
     .value_counts(normalize=True)
     .sort_index() * 100)
     .round(2)
)

# ------------------------------------------------------------
# Strong consensus
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("HIGH-CONFIDENCE ANOMALIES")
print("=" * 60)

high_confidence = results[
    results["Model_Agreement"] >= 3
]

print(
    "Observations flagged by at least 3/4 models:",
    len(high_confidence)
)

print(
    "Percentage:",
    round(
        len(high_confidence) / len(results) * 100,
        2
    ),
    "%"
)

# ------------------------------------------------------------
# All 4 models agree
# ------------------------------------------------------------

all_models = results[
    results["Model_Agreement"] == 4
]

print(
    "\nObservations flagged by ALL 4 models:",
    len(all_models)
)

print(
    "Percentage:",
    round(
        len(all_models) / len(results) * 100,
        2
    ),
    "%"
)

MODEL AGREEMENT

Number of models agreeing on anomaly:
Model_Agreement
0    9052
1     102
2      51
3      64
4      91
Name: count, dtype: int64

Percentage of dataset:
Model_Agreement
0    96.71
1     1.09
2     0.54
3     0.68
4     0.97
Name: proportion, dtype: float64

HIGH-CONFIDENCE ANOMALIES
Observations flagged by at least 3/4 models: 155
Percentage: 1.66 %

Observations flagged by ALL 4 models: 91
Percentage: 0.97 %
